# Tennis Sensor Analysis — Cloud Notebook

Processes raw EMG/IMU logs from the ESP32 Tennis Sensor system.

**New model (v2):** The entire log folder is treated as **one tennis session**.
All BLE connections are stitched into a single continuous timeline using wall-clock
timestamps. BLE reconnection boundaries are marked on all plots.

### Prerequisites:
1. **Google Drive** mounted (or local Jupyter with files accessible)
2. **Python Scripts** (`emg_util.py`, `emg_util_yaml_runner.py`, `capture_loader.py`,
   `capture_health.py`, `fatigue_monitor.py`) uploaded to a folder in Drive
3. **Log Files** — upload a ZIP or individual CSV files (`emg_log_*.csv`, `imu_log_*.csv`)

### Analysis modes (set in Cell 5):
| Mode | `FAST_MODE` | `MEDIUM_MODE` | Description |
|------|-------------|---------------|-------------|
| `both` (default) | False | False | Full per-swing EMG features (MDF, RMS, WL, ZC, SSC, MNF) |
| `medium` | False | True | MDF + RMS + envelope only — faster, key fatigue indicators |
| `bout` | True | — | IMU-only bout analysis, no EMG features |

### What this produces (all in `OUT_DIR/`, no subdirectories):
| File | Description | Mode |
|------|-------------|------|
| `capture_segments.csv` | BLE connection list with reconnection times | all |
| `swing_windows.csv` | All detected swings (full session) | all |
| `swing_features.csv` | Per-swing EMG features + fatigue deltas | both/medium |
| `windowed_rms.csv` | Windowed RMS/MAV across full session | medium |
| `bout_summary.csv` | Bout-level summary | all |
| `baseline.json` | Baseline EMG values (first N swings) | both/medium |
| `session_summary.json` | Total duration, swing count, reconnection times | all |
| `imu_activity_overlay.png` | IMU intensity + active spans + swing peaks | all |
| `fatigue_trend.png` | MDF/RMS over time with reconnection markers | both/medium |

In [ ]:
# --- CELL 1: SETUP & MOUNT DRIVE ---

import sys
import os

# Detect environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✅ Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('ℹ️  Running locally (not in Colab).')

# Install dependencies
if IN_COLAB:
    !pip install -q pyyaml scipy pandas matplotlib seaborn scikit-learn
else:
    print('ℹ️  Assuming dependencies already installed locally.')

print('✅ Setup complete.')

In [ ]:
# --- CELL 2: UNZIP DATA (Optional - skip if using individual CSVs) ---

import zipfile

# ========================================================================
# OPTION 1: If you uploaded a ZIP file, configure these paths:
# ========================================================================
USE_ZIP = True  # Set to False if you uploaded individual CSV files

if IN_COLAB:
    ZIP_PATH   = '/content/drive/MyDrive/Tennis_Project/logs_new.zip'  # ← Update this
    EXTRACT_TO = '/content/drive/MyDrive/Tennis_Project/raw_logs'      # ← Where to extract
else:
    ZIP_PATH   = r'C:/wangx/ftc/arduino/py_scripts/logs_new.zip'
    EXTRACT_TO = r'C:/wangx/ftc/arduino/py_scripts/logs_new'

if USE_ZIP:
    if os.path.exists(ZIP_PATH):
        print(f'📦 Found ZIP: {ZIP_PATH}')
        print(f'📂 Extracting to: {EXTRACT_TO}...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_TO)
        all_files = os.listdir(EXTRACT_TO)
        n_emg = len([f for f in all_files if f.startswith('emg_log_') and f.endswith('.csv')])
        n_imu = len([f for f in all_files if f.startswith('imu_log_') and f.endswith('.csv')])
        print(f'✅ Extracted {n_emg} EMG files, {n_imu} IMU files')
    else:
        print(f'⚠️  ZIP not found: {ZIP_PATH}')
        print('   Please update ZIP_PATH above or set USE_ZIP=False')
else:
    print('ℹ️  ZIP extraction disabled (USE_ZIP=False)')
    print('   Assuming CSV files are already in place.')

In [ ]:
# --- CELL 3: PATH CONFIGURATION ---
# Update these paths to match your setup.
# LOG_DIR should contain ALL BLE capture files for ONE tennis session.

if IN_COLAB:
    # === COLAB PATHS (update to match your Drive layout) ===
    SCRIPT_DIR = '/content/drive/MyDrive/Tennis_Project/scripts'
    LOG_DIR    = '/content/drive/MyDrive/Tennis_Project/raw_logs'   # ← folder with all BLE CSVs
    OUT_DIR    = '/content/drive/MyDrive/Tennis_Project/processed_out'
else:
    # === LOCAL PATHS (update to match your PC) ===
    SCRIPT_DIR = r'C:/wangx/ftc/emg_util'
    LOG_DIR    = r'C:/wangx/ftc/emg_util/log0207'
    OUT_DIR    = r'C:/wangx/ftc/emg_util/out0207'

os.makedirs(OUT_DIR, exist_ok=True)

# Add scripts to path
if os.path.exists(SCRIPT_DIR):
    if SCRIPT_DIR not in sys.path:
        sys.path.insert(0, SCRIPT_DIR)
    print(f'✅ Script dir: {SCRIPT_DIR}')
else:
    print(f'⚠️  Script dir NOT found: {SCRIPT_DIR}')
    print('   Update SCRIPT_DIR above to point to your .py files.')

if os.path.exists(LOG_DIR):
    all_files = os.listdir(LOG_DIR)
    n_emg = len([f for f in all_files if f.startswith('emg_log_') and f.endswith('.csv')])
    n_imu = len([f for f in all_files if f.startswith('imu_log_') and f.endswith('.csv')])
    print(f'✅ Log dir: {LOG_DIR}  ({n_emg} EMG files, {n_imu} IMU files)')
    # Count unique BLE connections (session_ids)
    import re
    session_ids_found = sorted(set(
        re.match(r'emg_log_(\d{8}_\d{6})_', f).group(1)
        for f in all_files
        if f.startswith('emg_log_') and re.match(r'emg_log_(\d{8}_\d{6})_', f)
    ))
    print(f'   BLE connections found: {len(session_ids_found)} → {session_ids_found}')
    print(f'   These will be stitched into ONE tennis session.')
else:
    print(f'⚠️  Log dir NOT found: {LOG_DIR}')
    print('   Update LOG_DIR above to point to your log files.')

In [ ]:
# --- CELL 4: IMPORTS ---

import json
import traceback
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import emg_util_yaml_runner as runner
    import capture_loader as loader
    import emg_util as core
    import capture_health as health
    print('✅ Tennis analysis modules imported.')
except ImportError as e:
    print(f'❌ Import error: {e}')
    print('   Check that SCRIPT_DIR is correct and all .py files are present.')
    raise

In [ ]:
# --- CELL 5: CONFIGURATION ---
# Mirrors config.yaml. No config file needed — everything is defined here.
# Set SESSION_LABEL to a human-readable name for this session (used in plot titles).

SESSION_LABEL = ''  # e.g. '2026-02-08 practice' — leave empty to use folder name

# --- Analysis mode ---
# FAST_MODE=True  : IMU-only bout analysis (no EMG features, fastest)
# MEDIUM_MODE=True: MDF + RMS + envelope per swing + windowed RMS (good balance)
# Both False      : Full per-swing EMG features (MDF, RMS, WL, ZC, SSC, MNF)
FAST_MODE   = False   # True = IMU-only bout analysis (skip all EMG features)
MEDIUM_MODE = False   # True = MDF+RMS+envelope only (skip WL/ZC/SSC/MNF, faster)
# Note: FAST_MODE takes priority over MEDIUM_MODE

if FAST_MODE:
    _mode = 'bout'
elif MEDIUM_MODE:
    _mode = 'medium'
else:
    _mode = 'both'

CLOUD_CONFIG = {
    'session': {
        'emg_glob': 'emg_log_*.csv',
        'imu_glob': 'imu_log_*.csv',
        'vitals_path': 'vitals_log.csv',
        'emg_fs_hz': 2000,
        'imu_fs_fallback_hz': 200,
        'use_physical_units': True,
        'acc_fsr_default_g': 8,      # matches current hardware: ±8g (was 4, then 2)
        'gyro_fsr_default_dps': 2000,  # matches current hardware: ±2000 dps (was 1000, then 250)
        'out_dir': OUT_DIR,
        'label': SESSION_LABEL,  # used in plot titles
    },
    'features': {
        'mode': _mode,  # 'bout' | 'medium' | 'both'
        'zc_thr': 0.01,
        'ssc_thr': 0.01,
        'win_rms_s': 10.0,   # windowed RMS window length in seconds (medium mode)
        'step_rms_s': 5.0,   # windowed RMS step in seconds (medium mode)
    },
    'swing_detection': {
        'use_gyro': True,
        'thr_gyro_dps': 300,       # fixed gyro threshold in dps (replaces z-score)
        'thr_acc_g': 0.0,          # optional acc gate in g (0=disabled)
        'min_gap_ms': 2000,        # min ms between swings
        'mdf_pad_ms': 300.0,
    },
    'baseline': {
        'mode': 'swing',
        'warmup_skip_min': 5.0,
        'warmup_skip_swings': 0,
        'baseline_swings': 20,
        'baseline_window_sec': 60.0,
    },
    'bout': {
        'gap_thr_s': 10,
        'gap_thr_long_s': 30,
        'min_bout_swings': 3,
        'rest_dur_s': 4,
        'merge_short_rest_s': 6,
    },
    'plots': {
        # Windowed EMG plots only in full mode (both); medium/bout skip core.plot_all
        'enable': not FAST_MODE and not MEDIUM_MODE,
        'full_duration': True,
        'emg_max_points': 300000,
        'do_fatigue': True,
        'win_sec': 1.0,
        'step_sec': 0.25,
    },
    'diagnostics': {
        'emg_imu_correlation': {
            # Correlation requires windowed features (full mode only)
            'enable': not FAST_MODE and not MEDIUM_MODE,
            'method': 'pearson',
        },
        'activity_overlay': {'enable': True, 'plot_swings': True, 'plot_signal': 'intensity'},
    },
    'activity_gating': {
        'enable': True,
        'w_gyro': 0.8,
        'w_acc': 0.2,
        'smooth_ms': 250.0,
        'p_high': 90,
        'p_low': 75,
        'expand_ms': 400,
        'min_active_ms': 1000,
        'merge_gap_ms': 2000,
        'skip_min': 0,
        'filter_swings': False,
    },
    'health': {
        'enable': not FAST_MODE,  # EMG health check runs in both + medium mode
        'window_s': 1.0,
        'bad_persist_s': 2.0,
        'emg_zero_frac_bad': 0.9,
        'emg_flat_std_bad': 1.0,
        'emg_gap_bad_us': 200000,
        'filter_swings_by_emg_bad': True,
    },
}

_mode_desc = {
    'bout':   'IMU-only bout analysis (no EMG features)',
    'medium': 'MDF + RMS + envelope per swing + windowed RMS',
    'both':   'Full per-swing EMG features (MDF, RMS, WL, ZC, SSC, MNF)',
}
print('✅ Configuration loaded.')
print(f'   Mode: {_mode} — {_mode_desc[_mode]}')
print(f'   EMG fs: {CLOUD_CONFIG["session"]["emg_fs_hz"]} Hz')
print(f'   EMG health filtering: {"DISABLED (fast mode)" if FAST_MODE else "ENABLED"}')
print(f'   Session label: "{SESSION_LABEL or "(folder name)"}')
print(f'   Output: {OUT_DIR}')

In [ ]:
# --- CELL 6: RUN TENNIS SESSION ANALYSIS ---
# Analyzes the entire LOG_DIR as ONE tennis session.
# All BLE connections are stitched into a single continuous timeline.
# Outputs go directly to OUT_DIR/ (no subdirectories).

import traceback

try:
    out_path = runner.analyze_tennis_session(LOG_DIR, CLOUD_CONFIG, OUT_DIR)
    print(f'\n✅ Analysis complete → {out_path}')
    # Show what was produced
    produced = sorted(os.listdir(out_path))
    print(f'\nOutput files ({len(produced)}):')
    for f in produced:
        fpath = os.path.join(out_path, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f'  {f:45s}  {size_kb:6.1f} KB')
except Exception as e:
    print(f'❌ Analysis failed:')
    traceback.print_exc()

In [ ]:
# --- CELL 7: LOAD RESULTS ---
# Load the outputs from the flat OUT_DIR/ (no subdirectories).

# Session summary
summary_path = os.path.join(OUT_DIR, 'session_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        SESSION_SUMMARY = json.load(f)
    print('=== Session Summary ===')
    for k, v in SESSION_SUMMARY.items():
        print(f'  {k}: {v}')
else:
    SESSION_SUMMARY = {}
    print('⚠️  session_summary.json not found — run Cell 6 first.')

# Capture segments (BLE connections)
cap_path = os.path.join(OUT_DIR, 'capture_segments.csv')
CAPTURE_SEGS = pd.read_csv(cap_path) if os.path.exists(cap_path) else pd.DataFrame()
if len(CAPTURE_SEGS):
    print(f'\n=== BLE Connections ({len(CAPTURE_SEGS)}) ===')
    print(CAPTURE_SEGS.to_string(index=False))

# Swing features (both + medium mode)
sf_path = os.path.join(OUT_DIR, 'swing_features.csv')
SWING_DF = pd.read_csv(sf_path) if os.path.exists(sf_path) else pd.DataFrame()
print(f'\nSwing features: {len(SWING_DF)} swings')
if len(SWING_DF):
    print(SWING_DF.describe().round(3))

# Windowed RMS (medium mode only)
wrms_path = os.path.join(OUT_DIR, 'windowed_rms.csv')
WINDOWED_RMS_DF = pd.read_csv(wrms_path) if os.path.exists(wrms_path) else pd.DataFrame()
if len(WINDOWED_RMS_DF):
    t0_s = WINDOWED_RMS_DF['t_ms'].min() / 1000.0
    t1_s = WINDOWED_RMS_DF['t_ms'].max() / 1000.0
    print(f'\nWindowed RMS: {len(WINDOWED_RMS_DF)} windows '
          f'(t={t0_s:.1f}s — {t1_s:.1f}s, span={t1_s-t0_s:.0f}s)')

# Bout summary
bout_path = os.path.join(OUT_DIR, 'bout_summary.csv')
BOUTS_DF = pd.read_csv(bout_path) if os.path.exists(bout_path) else pd.DataFrame()
print(f'\nBout summary: {len(BOUTS_DF)} bouts')
if len(BOUTS_DF):
    print(BOUTS_DF.to_string(index=False))

In [ ]:
# --- CELL 8: VISUALIZE FATIGUE PROGRESSION ---
# Plots MDF, RMS, cost index, and MDF drop % across the full session.
# BLE reconnection boundaries are shown as vertical dashed lines.

if len(SWING_DF) == 0:
    print('No swing data available. Run previous cells first (requires features.mode="both" or "medium").')
else:
    df = SWING_DF.copy()
    df['t_min'] = df['t_peak_ms'] / 60000.0
    df['cost_index'] = df['rms'] / df['imu_peak_mag'].replace(0, np.nan).fillna(1)

    # BLE reconnection times in minutes
    reconnection_min = [
        t / 60.0 for t in SESSION_SUMMARY.get('reconnection_times_s', [])
    ]

    def add_reconnection_lines(ax):
        for rt in reconnection_min:
            ax.axvline(rt, color='navy', ls='--', lw=1.0, alpha=0.6)
            ax.text(rt, ax.get_ylim()[1], f' {rt:.1f}m',
                    fontsize=7, color='navy', va='top', rotation=90)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f'Fatigue Analysis — {SESSION_LABEL or os.path.basename(LOG_DIR)}\n'
        f'({SESSION_SUMMARY.get("num_swings", "?")}) swings, '
        f'{SESSION_SUMMARY.get("total_duration_min", "?"):.1f} min, '
        f'{len(CAPTURE_SEGS)} BLE connections',
        fontsize=12, fontweight='bold'
    )

    # 1. MDF over time
    ax = axes[0, 0]
    if 'mdf' in df.columns:
        ax.scatter(df['t_min'], df['mdf'], s=8, alpha=0.5, color='steelblue', label='MDF')
        # Rolling median trend
        win = min(20, max(3, len(df) // 10))
        trend = df['mdf'].rolling(win, center=True, min_periods=3).median()
        ax.plot(df['t_min'], trend, lw=1.5, color='steelblue', label='trend')
    add_reconnection_lines(ax)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('MDF (Hz)')
    ax.set_title('Median Frequency (↓ = fatigue)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # 2. RMS over time
    ax = axes[0, 1]
    if 'rms' in df.columns:
        ax.scatter(df['t_min'], df['rms'], s=8, alpha=0.5, color='darkorange', label='RMS')
        trend = df['rms'].rolling(win, center=True, min_periods=3).median()
        ax.plot(df['t_min'], trend, lw=1.5, color='darkorange', label='trend')
    add_reconnection_lines(ax)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('RMS')
    ax.set_title('EMG RMS (↑ = more effort / fatigue)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # 3. Cost Index scatter (color = MDF)
    ax = axes[1, 0]
    if 'cost_index' in df.columns and 'mdf' in df.columns:
        sc = ax.scatter(df['t_min'], df['cost_index'],
                        c=df['mdf'], cmap='RdYlGn', alpha=0.6, s=20)
        plt.colorbar(sc, ax=ax, label='MDF (Hz)')
    add_reconnection_lines(ax)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('Cost Index (RMS / SwingSpeed)')
    ax.set_title('Neuromechanical Cost (color = MDF)')
    ax.grid(True, alpha=0.3)

    # 4. MDF drop % from baseline
    ax = axes[1, 1]
    mdf_col = 'swing_mdf_drop_pct' if 'swing_mdf_drop_pct' in df.columns else 'mdf_drop_pct'
    if mdf_col in df.columns:
        ax.scatter(df['t_min'], df[mdf_col], s=8, alpha=0.5, color='purple', label='MDF drop %')
        trend = df[mdf_col].rolling(win, center=True, min_periods=3).median()
        ax.plot(df['t_min'], trend, lw=1.5, color='purple', label='trend')
        ax.axhline(12, color='r', ls='--', lw=1, label='12% threshold')
        ax.set_ylabel('MDF Drop (%)')
        ax.set_title('MDF Drop % from Baseline')
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'mdf_drop_pct not available\n(requires features.mode="both" or "medium")',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
    add_reconnection_lines(ax)
    ax.set_xlabel('Time (min)'); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    dash_path = os.path.join(OUT_DIR, 'dashboard_fatigue.png')
    fig.savefig(dash_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {dash_path}')

In [ ]:
# --- CELL 9: BOUT SUMMARY PLOT ---

if len(BOUTS_DF) == 0:
    print('No bout data available. Run previous cells first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Bout Summary — {SESSION_LABEL or os.path.basename(LOG_DIR)}', fontsize=12)

    # Swings per bout
    ax = axes[0]
    ax.bar(BOUTS_DF['bout_id'], BOUTS_DF['num_swings'], color='steelblue', alpha=0.8)
    ax.set_xlabel('Bout ID'); ax.set_ylabel('Swings')
    ax.set_title('Swings per Bout')
    ax.grid(True, alpha=0.3, axis='y')

    # Swings per minute per bout
    ax = axes[1]
    ax.bar(BOUTS_DF['bout_id'], BOUTS_DF['swings_per_min'], color='darkorange', alpha=0.8)
    ax.set_xlabel('Bout ID'); ax.set_ylabel('Swings / min')
    ax.set_title('Swing Rate per Bout')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    bout_plot_path = os.path.join(OUT_DIR, 'dashboard_bouts.png')
    fig.savefig(bout_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {bout_plot_path}')
    print(f'\nTotal: {BOUTS_DF["num_swings"].sum()} swings across {len(BOUTS_DF)} bouts')
    print(f'Avg swing rate: {BOUTS_DF["swings_per_min"].mean():.1f} swings/min')